# Job Recommendation System
## Step 4: Recommendation Engine (Content-Based Filtering)

**Internship Project** | Gamage Recruiters  
**Focus:** Recruitment & HR / Data Science & Machine Learning

---

In this step we build the **core recommendation engine** using Content-Based Filtering.
The engine compares each candidate's profile against every job posting and returns a ranked list of the best matches.

### Scoring Formula

$$\text{Final Score} = \text{TF-IDF Cosine Similarity} + \text{Experience Bonus} + \text{Domain Bonus} + \text{Location Bonus}$$

| Component | Weight | Description |
|---|---|---|
| TF-IDF Cosine Similarity | up to ~0.75 | Skill & text overlap between candidate and job |
| Experience Bonus | +0.15 / +0.07 / 0 | Exact level match / 1 level off / 2+ levels off |
| Domain Bonus | +0.10 | Candidate domain matches job industry |
| Location Bonus | +0.05 | Job location is in candidate's preferred list |

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics.pairwise import cosine_similarity

print('Libraries loaded successfully')

Libraries loaded successfully


In [15]:
import os

# Create directories if they don't exist
os.makedirs('../outputs', exist_ok=True)

## 2. Load All Preprocessed Artifacts from Step 3

In [3]:
# Dataframes
df_c = pd.read_csv('../data/candidates_preprocessed.csv')
df_j = pd.read_csv('../data/job_postings_preprocessed.csv')

# TF-IDF matrices
candidate_tfidf = sp.load_npz('../models/candidate_tfidf_matrix.npz')
job_tfidf       = sp.load_npz('../models/job_tfidf_matrix.npz')

# Encoders & scalers
with open('../models/encoders.pkl', 'rb') as f:
    encoders = pickle.load(f)

exp_order = encoders['exp_order']

print(f'Candidates       : {df_c.shape[0]} rows x {df_c.shape[1]} cols')
print(f'Job Postings     : {df_j.shape[0]} rows x {df_j.shape[1]} cols')
print(f'Candidate TF-IDF : {candidate_tfidf.shape}')
print(f'Job TF-IDF       : {job_tfidf.shape}')

Candidates       : 500 rows x 22 cols
Job Postings     : 200 rows x 26 cols
Candidate TF-IDF : (500, 300)
Job TF-IDF       : (200, 300)


## 3. Step 1 — TF-IDF Cosine Similarity

Cosine similarity measures the **angle** between two TF-IDF vectors. A score of 1.0 means identical skill profiles; 0.0 means no overlap at all.

The result is a **(500 x 200)** matrix — one similarity score for every candidate-job pair.

In [4]:
sim_matrix = cosine_similarity(candidate_tfidf, job_tfidf)   # shape: (500, 200)

print(f'Similarity matrix shape : {sim_matrix.shape}')
print(f'Score range             : {sim_matrix.min():.4f} - {sim_matrix.max():.4f}')
print(f'Mean similarity score   : {sim_matrix.mean():.4f}')
print(f'\nSample scores (candidate 0 vs first 5 jobs):')
for k in range(5):
    print(f'  vs {df_j["job_title"].iloc[k]:35s} [{df_j["industry"].iloc[k]:12s}] -> {sim_matrix[0, k]:.4f}')

Similarity matrix shape : (500, 200)
Score range             : 0.0000 - 0.7578
Mean similarity score   : 0.0716

Sample scores (candidate 0 vs first 5 jobs):
  vs Financial Analyst                   [Finance     ] -> 0.0000
  vs Full Stack Developer                [Technology  ] -> 0.2372
  vs Operations Manager                  [Operations  ] -> 0.0136
  vs Project Manager                     [Operations  ] -> 0.0000
  vs Compensation Analyst                [HR          ] -> 0.0000


## 4. Step 2 — Experience Level Bonus

A candidate who is a perfect experience-level match gets an extra boost. Someone one level off gets a smaller boost. This prevents a Senior-level candidate from being recommended Entry-level roles just because skills overlap.

In [5]:
c_exp = df_c['experience_encoded'].values.reshape(-1, 1)   # (500, 1)
j_exp = df_j['experience_encoded'].values.reshape(1, -1)   # (1, 200)
exp_diff = np.abs(c_exp - j_exp)                           # (500, 200) broadcasted

exp_bonus = np.where(exp_diff == 0, 0.15,         # exact match
            np.where(exp_diff == 1, 0.07, 0.0))   # 1 level off

print('Experience bonus matrix shape :', exp_bonus.shape)
print('Bonus values: 0.15 (exact match) | 0.07 (+-1 level) | 0.0 (2+ levels off)')
print(f'\nBonus distribution:')
unique, counts = np.unique(exp_bonus, return_counts=True)
for v, c in zip(unique, counts):
    print(f'  Bonus {v:.2f} -> {c:,} pairs ({c/(500*200)*100:.1f}%)')

Experience bonus matrix shape : (500, 200)
Bonus values: 0.15 (exact match) | 0.07 (+-1 level) | 0.0 (2+ levels off)

Bonus distribution:
  Bonus 0.00 -> 55,844 pairs (55.8%)
  Bonus 0.07 -> 27,483 pairs (27.5%)
  Bonus 0.15 -> 16,673 pairs (16.7%)


## 5. Step 3 — Domain / Industry Match Bonus

A candidate in Technology should see Technology jobs ranked higher than Finance jobs, even if the skill text partially overlaps.

In [6]:
c_dom = df_c['domain_encoded'].values.reshape(-1, 1)    # (500, 1)
j_dom = df_j['industry_encoded'].values.reshape(1, -1)  # (1, 200)

dom_bonus = np.where(c_dom == j_dom, 0.10, 0.0)         # (500, 200)

print('Domain bonus matrix shape :', dom_bonus.shape)
print('Bonus value               : +0.10 when candidate domain == job industry')
n_matched = (dom_bonus > 0).sum()
print(f'Domain-matched pairs      : {n_matched:,} / {500*200:,} ({n_matched/(500*200)*100:.1f}%)')

Domain bonus matrix shape : (500, 200)
Bonus value               : +0.10 when candidate domain == job industry
Domain-matched pairs      : 20,000 / 100,000 (20.0%)


## 6. Step 4 — Location Preference Bonus

Each candidate listed 1-3 preferred locations in Step 2. If a job's location appears in that list, the job gets a small boost.

In [7]:
def location_bonus_score(pref_locs_str, job_loc):
    """Return 0.05 if job location matches any of the candidate's preferred locations."""
    if pd.isna(pref_locs_str):
        return 0.0
    prefs = [l.strip().lower() for l in str(pref_locs_str).split(',')]
    if job_loc.strip().lower() in prefs:
        return 0.05
    return 0.0

# Build (500 x 200) location bonus matrix
loc_bonus = np.zeros((len(df_c), len(df_j)))
for i, pref in enumerate(df_c['preferred_locations']):
    for k, jloc in enumerate(df_j['location']):
        loc_bonus[i, k] = location_bonus_score(pref, jloc)

print('Location bonus matrix shape :', loc_bonus.shape)
print('Bonus value                 : +0.05 when job location in candidate preferences')
n_loc = (loc_bonus > 0).sum()
print(f'Location-matched pairs      : {n_loc:,} / {500*200:,} ({n_loc/(500*200)*100:.1f}%)')

Location bonus matrix shape : (500, 200)
Bonus value                 : +0.05 when job location in candidate preferences
Location-matched pairs      : 9,756 / 100,000 (9.8%)


## 7. Step 5 — Compute Final Scores

All four components are summed into a single **final relevance score** per candidate-job pair.

In [8]:
final_scores = sim_matrix + exp_bonus + dom_bonus + loc_bonus   # (500, 200)

print(f'Final score matrix shape : {final_scores.shape}')
print(f'Score range              : {final_scores.min():.4f} - {final_scores.max():.4f}')
print(f'Mean final score         : {final_scores.mean():.4f}')
print()
print('Score component breakdown (averages):')
print(f'  TF-IDF cosine  : {sim_matrix.mean():.4f}')
print(f'  Exp bonus      : {exp_bonus.mean():.4f}')
print(f'  Domain bonus   : {dom_bonus.mean():.4f}')
print(f'  Location bonus : {loc_bonus.mean():.4f}')
print(f'  -----------------------------------------')
print(f'  Final (sum)    : {final_scores.mean():.4f}')

Final score matrix shape : (500, 200)
Score range              : 0.0000 - 0.9660
Mean final score         : 0.1408

Score component breakdown (averages):
  TF-IDF cosine  : 0.0716
  Exp bonus      : 0.0442
  Domain bonus   : 0.0200
  Location bonus : 0.0049
  -----------------------------------------
  Final (sum)    : 0.1408


## 8. Skill Overlap Helper

Beyond the score, we compute **skill overlap %** — what fraction of the job's required skills the candidate already has. This is a human-readable quality signal shown in the results table.

In [9]:
def skill_overlap_pct(cand_skills_str, job_skills_str):
    """Percentage of job's required skills that the candidate possesses."""
    if pd.isna(cand_skills_str) or pd.isna(job_skills_str):
        return 0.0
    c_skills = set(s.strip().lower() for s in str(cand_skills_str).split(','))
    j_skills = set(s.strip().lower() for s in str(job_skills_str).split(','))
    if not j_skills:
        return 0.0
    return round(len(c_skills & j_skills) / len(j_skills) * 100, 1)

# Demo
c0_skills = df_c['skills'].iloc[1]
j0_skills = df_j['required_skills'].iloc[0]
print(f'Candidate skills : {c0_skills}')
print(f'Job req. skills  : {j0_skills}')
print(f'Overlap          : {skill_overlap_pct(c0_skills, j0_skills)}%')

Candidate skills : R, Tax Planning, Attention to Detail, Financial Modeling, Portfolio Management, Tableau, Equity Research, GAAP, Analytical Thinking, Risk Analysis, Accounting
Job req. skills  : Teamwork, Financial Modeling, Tax Planning, Tableau, Accounting, Python, Fixed Income, Presentation Skills, Power BI
Overlap          : 44.4%


## 9. Generate Top-N Recommendations for All Candidates

In [10]:
N = 5   # Number of recommendations per candidate

rows = []
for i in range(len(df_c)):
    top_idx = np.argsort(final_scores[i])[::-1][:N]
    cand = df_c.iloc[i]
    for rank, idx in enumerate(top_idx, 1):
        job = df_j.iloc[idx]
        rows.append({
            'candidate_id':       cand['candidate_id'],
            'candidate_name':     cand['name'],
            'candidate_domain':   cand['primary_domain'],
            'candidate_level':    cand['experience_level'],
            'candidate_location': cand['location'],
            'rank':               rank,
            'job_id':             job['job_id'],
            'job_title':          job['job_title'],
            'company':            job['company_name'],
            'industry':           job['industry'],
            'job_location':       job['location'],
            'job_level':          job['experience_level'],
            'salary_min_lkr':     job['salary_min_lkr'],
            'salary_max_lkr':     job['salary_max_lkr'],
            'tfidf_score':        round(sim_matrix[i, idx], 4),
            'exp_bonus':          round(exp_bonus[i, idx], 4),
            'dom_bonus':          round(dom_bonus[i, idx], 4),
            'loc_bonus':          round(loc_bonus[i, idx], 4),
            'final_score':        round(final_scores[i, idx], 4),
            'skill_overlap_pct':  skill_overlap_pct(cand['skills'], job['required_skills'])
        })

df_recs = pd.DataFrame(rows)
print(f'Recommendations generated')
print(f'  Total rows (500 candidates x {N} recs) : {len(df_recs)}')
print(f'  Columns : {list(df_recs.columns)}')

Recommendations generated
  Total rows (500 candidates x 5 recs) : 2500
  Columns : ['candidate_id', 'candidate_name', 'candidate_domain', 'candidate_level', 'candidate_location', 'rank', 'job_id', 'job_title', 'company', 'industry', 'job_location', 'job_level', 'salary_min_lkr', 'salary_max_lkr', 'tfidf_score', 'exp_bonus', 'dom_bonus', 'loc_bonus', 'final_score', 'skill_overlap_pct']


## 10. Inspect Results — Sample Candidates

In [11]:
def show_recommendations(candidate_id, df_recs, df_c):
    """Pretty-print top-N recommendations for a given candidate."""
    cand = df_c[df_c['candidate_id'] == candidate_id].iloc[0]
    recs = df_recs[df_recs['candidate_id'] == candidate_id].copy()
    print('=' * 75)
    print(f'Candidate: {cand["name"]}  [{candidate_id}]')
    print(f'  Domain   : {cand["primary_domain"]}  |  Level : {cand["experience_level"]}  |  Location : {cand["location"]}')
    print(f'  Skills   : {cand["skills"]}')
    print(f'  Prefers  : {cand["preferred_locations"]}')
    print('-' * 75)
    print(f'  {"Rank":<5} {"Job Title":<35} {"Company":<25} {"Score":<7} {"Overlap"}')
    print('-' * 75)
    for _, row in recs.iterrows():
        print(f'  #{int(row["rank"]):<4} {row["job_title"]:<35} {row["company"]:<25} {row["final_score"]:<7.4f} {row["skill_overlap_pct"]}%')
        print(f'        Location: {row["job_location"]}  |  Level: {row["job_level"]}  |  LKR {row["salary_min_lkr"]:,} - {row["salary_max_lkr"]:,}')
        print(f'        Scores -> TF-IDF: {row["tfidf_score"]:.4f}  Exp: +{row["exp_bonus"]:.2f}  Domain: +{row["dom_bonus"]:.2f}  Location: +{row["loc_bonus"]:.2f}')
    print()

# Show one candidate from each of 3 domains
sample_ids = (
    df_c[df_c['primary_domain'] == 'Technology']['candidate_id'].iloc[0],
    df_c[df_c['primary_domain'] == 'Finance']['candidate_id'].iloc[0],
    df_c[df_c['primary_domain'] == 'HR']['candidate_id'].iloc[0],
)
for cid in sample_ids:
    show_recommendations(cid, df_recs, df_c)

Candidate: Nuwan Rajapaksa  [C0001]
  Domain   : Technology  |  Level : Junior  |  Location : Polonnaruwa
  Skills   : MongoDB, scikit-learn, PyTorch, TensorFlow
  Prefers  : Polonnaruwa
---------------------------------------------------------------------------
  Rank  Job Title                           Company                   Score   Overlap
---------------------------------------------------------------------------
  #1    NLP Engineer                        Calcey Technologies       0.4874  11.1%
        Location: Nuwara Eliya  |  Level: Lead  |  LKR 281,300 - 365,191
        Scores -> TF-IDF: 0.3874  Exp: +0.00  Domain: +0.10  Location: +0.00
  #2    Full Stack Developer                IFS R&D International     0.4872  33.3%
        Location: Kandy  |  Level: Junior  |  LKR 61,467 - 91,839
        Scores -> TF-IDF: 0.2372  Exp: +0.15  Domain: +0.10  Location: +0.00
  #3    Business Intelligence Analyst       Zone24x7                  0.4723  16.7%
        Location: Matara  |  L

## 11. Wide-Format Summary Table (One Row Per Candidate)

In [12]:
wide_rows = []
for cid, grp in df_recs.groupby('candidate_id'):
    cand = df_c[df_c['candidate_id'] == cid].iloc[0]
    row = {
        'candidate_id':   cid,
        'candidate_name': cand['name'],
        'domain':         cand['primary_domain'],
        'level':          cand['experience_level'],
    }
    for _, rec in grp.iterrows():
        r = int(rec['rank'])
        row[f'Rank{r}_title']   = rec['job_title']
        row[f'Rank{r}_company'] = rec['company']
        row[f'Rank{r}_score']   = rec['final_score']
    wide_rows.append(row)

df_wide = pd.DataFrame(wide_rows)
print(f'Wide table shape : {df_wide.shape}')
df_wide.head(10)

Wide table shape : (500, 19)


,candidate_id,candidate_name,domain,level,Rank1_title,Rank1_company,Rank1_score,Rank2_title,Rank2_company,Rank2_score,Rank3_title,Rank3_company,Rank3_score,Rank4_title,Rank4_company,Rank4_score,Rank5_title,Rank5_company,Rank5_score
0,C0001,Nuwan Rajapaksa,Technology,Junior,NLP Engineer,Calcey Technologies,0.4874,Full Stack Developer,IFS R&D International,0.4872,Business Intelligence Analyst,Zone24x7,0.4723,DevOps Engineer,Pearson Lanka,0.4305,DevOps Engineer,Codegen International,0.4258
1,C0002,Shankar Samarasinghe,Finance,Mid,Financial Analyst,Pan Asia Bank,0.7540,Auditor,LB Finance,0.7032,Credit Analyst,LOLC Finance,0.6867,Portfolio Manager,Bank of Ceylon,0.6614,Risk Analyst,People's Bank,0.6468
2,C0003,Hashan Seneviratne,Operations,Mid,Quality Assurance Analyst,Lanka Logistics,0.7268,Procurement Officer,CMA CGM Lanka,0.6946,Operations Analyst,SriLankan Airlines,0.6692,Quality Assurance Analyst,Holcim Lanka,0.6469,Process Improvement Specialist,Lanka Logistics,0.6425
3,C0004,Harshani Razik,Operations,Entry,Logistics Coordinator,Sri Lanka Ports Authority,0.5747,Procurement Officer,SriLankan Airlines,0.5600,Operations Analyst,Holcim Lanka,0.5375,ERP Consultant,Maersk Lanka,0.5214,Operations Manager,Expolanka Holdings,0.5155
4,C0005,Anjali Farook,Finance,Lead,Treasury Analyst,Nations Trust Bank,0.6894,Compliance Officer,Softlogic Finance,0.5161,Financial Controller,Sampath Bank,0.5140,Investment Analyst,LB Finance,0.4970,Accountant,LB Finance,0.4703
5,C0006,Ahamed Farook,Operations,Junior,Logistics Coordinator,Holcim Lanka,0.8755,Operations Manager,SriLankan Airlines,0.8435,Supply Chain Analyst,Lanka Logistics,0.7562,Quality Assurance Analyst,Holcim Lanka,0.6922,Procurement Officer,Airport & Aviation Services,0.6534
6,C0007,Fathima Sheriff,Operations,Entry,Procurement Officer,SriLankan Airlines,0.7690,Supply Chain Analyst,Lanka Logistics,0.5821,Logistics Coordinator,Holcim Lanka,0.5696,Logistics Coordinator,Sri Lanka Ports Authority,0.5337,Operations Analyst,SriLankan Airlines,0.5292
7,C0008,Kasun Wickramasinghe,Marketing,Entry,Content Writer,Leo Burnett Sri Lanka,0.8493,Growth Hacker,John Keells Holdings,0.7637,Growth Hacker,Dilmah Tea,0.7380,Marketing Analyst,Cargills (Ceylon),0.7295,SEO Analyst,John Keells Holdings,0.7266
8,C0009,Udara Amarasinghe,Finance,Junior,Accountant,Cargills Bank,0.7315,Auditor,Sampath Bank,0.6939,Credit Analyst,LOLC Finance,0.6683,Portfolio Manager,DFCC Bank,0.5958,Financial Analyst,Pan Asia Bank,0.5706
9,C0010,Thilini Perera,Finance,Senior,Accountant,LB Finance,0.6197,Treasury Analyst,Nations Trust Bank,0.6055,Financial Controller,LOLC Finance,0.5888,Treasury Analyst,Hatton National Bank (HNB),0.5847,Risk Analyst,People's Bank,0.5531


## 12. Score Distribution Analysis

In [13]:
print('=== Score Distribution - Top-1 Recommendations ===')
top1 = df_recs[df_recs['rank'] == 1]
print(top1['final_score'].describe().round(4))
print()
print('=== Skill Overlap % - Top-1 Recommendations ===')
print(top1['skill_overlap_pct'].describe().round(2))
print()
print('=== Domain Match Rate ===')
top1_match = (top1['candidate_domain'] == top1['industry']).mean()
print(f'{top1_match*100:.1f}% of top-1 recommendations are in the candidate domain')
print()
print('=== Experience Level Match Rate - Top-1 ===')
top1_exp = (top1['candidate_level'] == top1['job_level']).mean()
print(f'{top1_exp*100:.1f}% of top-1 recommendations exactly match candidate experience level')
print()
print('=== Top Recommended Companies ===')
print(df_recs['company'].value_counts().head(10).to_string())
print()
print('=== Top Recommended Job Titles ===')
print(df_recs['job_title'].value_counts().head(10).to_string())

=== Score Distribution - Top-1 Recommendations ===
count    500.0000
mean       0.6895
std        0.1013
min        0.4412
25%        0.6205
50%        0.6954
75%        0.7564
max        0.9660
Name: final_score, dtype: float64

=== Skill Overlap % - Top-1 Recommendations ===
count    500.00
mean      47.45
std       18.81
min        0.00
25%       33.30
50%       44.40
75%       60.00
max      100.00
Name: skill_overlap_pct, dtype: float64

=== Domain Match Rate ===
99.8% of top-1 recommendations are in the candidate domain

=== Experience Level Match Rate - Top-1 ===
69.4% of top-1 recommendations exactly match candidate experience level

=== Top Recommended Companies ===
company
John Keells Holdings    136
Cargills (Ceylon)       110
Aitken Spence           110
Hemas Holdings           97
Lanka Logistics          87
Virtusa                  85
Holcim Lanka             82
Dilmah Tea               81
JWT Sri Lanka            77
Expolanka Holdings       76

=== Top Recommended Job Tit

## 13. Reusable Query Function

A clean function to retrieve recommendations for **any candidate by ID** — useful for testing and the Step 8 dashboard.

In [14]:
def get_recommendations(candidate_id, top_n=5):
    """
    Return a DataFrame of top-N job recommendations for a given candidate_id.

    Parameters
    ----------
    candidate_id : str   e.g. 'C0042'
    top_n        : int   number of recommendations to return (default 5)

    Returns
    -------
    DataFrame with ranked job recommendations and all score components
    """
    cand_rows = df_c[df_c['candidate_id'] == candidate_id]
    if cand_rows.empty:
        print(f'Candidate {candidate_id} not found.')
        return None
    i = cand_rows.index[0]

    scores  = final_scores[i]
    top_idx = np.argsort(scores)[::-1][:top_n]
    cand_skills = cand_rows['skills'].values[0]

    results = []
    for rank, idx in enumerate(top_idx, 1):
        job = df_j.iloc[idx]
        results.append({
            'rank':              rank,
            'job_id':            job['job_id'],
            'job_title':         job['job_title'],
            'company':           job['company_name'],
            'industry':          job['industry'],
            'location':          job['location'],
            'job_level':         job['experience_level'],
            'salary_lkr':        f"LKR {job['salary_min_lkr']:,} - {job['salary_max_lkr']:,}",
            'required_skills':   job['required_skills'],
            'tfidf_score':       round(sim_matrix[i, idx], 4),
            'exp_bonus':         round(exp_bonus[i, idx], 4),
            'dom_bonus':         round(dom_bonus[i, idx], 4),
            'loc_bonus':         round(loc_bonus[i, idx], 4),
            'final_score':       round(scores[idx], 4),
            'skill_overlap_pct': skill_overlap_pct(cand_skills, job['required_skills'])
        })
    return pd.DataFrame(results)


# Try it on any candidate
test_id = df_c['candidate_id'].iloc[99]
cand_info = df_c[df_c['candidate_id'] == test_id].iloc[0]
print(f'Recommendations for: {cand_info["name"]} ({cand_info["primary_domain"]}, {cand_info["experience_level"]})')
print(f'Skills: {cand_info["skills"]}')
print()
get_recommendations(test_id, top_n=5)

Recommendations for: Roshan Ahamed (Technology, Manager)
Skills: Jira, Tableau, R, Computer Vision, Linux, Kotlin



,rank,job_id,job_title,company,industry,location,job_level,salary_lkr,required_skills,tfidf_score,exp_bonus,dom_bonus,loc_bonus,final_score,skill_overlap_pct
0,1,J0013,Data Scientist,WSO2,Technology,Batticaloa,Senior,"LKR 200,239 - 268,329","Tableau, C#, Jira",0.5348,0.00,0.1,0.00,0.6348,66.7
1,2,J0016,Mobile Developer,Pearson Lanka,Technology,Trincomalee,Entry,"LKR 37,774 - 47,671","MySQL, C++, Deep Learning, Tableau, Linux",0.4428,0.00,0.1,0.00,0.5428,40.0
2,3,J0058,AI Researcher,Surge Global,Technology,Vavuniya,Mid,"LKR 100,297 - 153,843","Git, REST API, Kotlin, Tableau, Machine Learni...",0.2972,0.00,0.1,0.00,0.3972,33.3
3,4,J0043,Business Intelligence Analyst,Virtusa,Technology,Badulla,Lead,"LKR 276,186 - 367,300","CI/CD, GCP, Communication, NLP, Jira, Machine ...",0.1653,0.07,0.1,0.05,0.3853,11.1
4,5,J0135,Business Intelligence Analyst,Axiata Digital Labs,Technology,Matara,Entry,"LKR 41,754 - 55,801","Deep Learning, Jira, Tableau, Redis, TensorFlo...",0.2833,0.00,0.1,0.00,0.3833,33.3


## 14. Save All Outputs

In [17]:
# Long format — full detail per candidate-job pair
df_recs.to_csv('../outputs/recommendations.csv', index=False)

# Wide format — one row per candidate, easy to read
df_wide.to_csv('../outputs/recommendations_wide.csv', index=False)

# Save score matrices for Step 7 (evaluation)
np.save('../models/final_score_matrix.npy', final_scores)
np.save('../models/tfidf_sim_matrix.npy',   sim_matrix)

print('All outputs saved:')
print('  ../outputs/recommendations.csv          (2500 rows - long format, full score breakdown)')
print('  ../outputs/recommendations_wide.csv     (500 rows  - wide format, one row per candidate)')
print('  ../models/final_score_matrix.npy       (500 x 200 - combined scores)')
print('  ../models/tfidf_sim_matrix.npy         (500 x 200 - cosine similarity only)')

All outputs saved:
  ../outputs/recommendations.csv          (2500 rows - long format, full score breakdown)
  ../outputs/recommendations_wide.csv     (500 rows  - wide format, one row per candidate)
  ../models/final_score_matrix.npy       (500 x 200 - combined scores)
  ../models/tfidf_sim_matrix.npy         (500 x 200 - cosine similarity only)


## 15. Summary

| Output | Description |
|---|---|
| `recommendations.csv` | 2,500 rows — top 5 jobs per candidate with full score breakdown |
| `recommendations_wide.csv` | 500 rows — one row per candidate, Rank1 through Rank5 columns |
| `final_score_matrix.npy` | 500 x 200 numpy array of final combined scores |
| `tfidf_sim_matrix.npy` | 500 x 200 cosine similarity scores (TF-IDF only) |

### Scoring Formula Recap

```
Final Score = TF-IDF Cosine Similarity        (main signal — skill & text match)
            + Experience Bonus (0.15/0.07/0)  (penalises wrong seniority level)
            + Domain Bonus (0.10/0)           (boosts same-industry jobs)
            + Location Bonus (0.05/0)         (boosts preferred locations)
```

### Design Decisions
- **Vectorised numpy operations** for experience and domain bonuses — fast even for large datasets
- **Skill overlap %** is computed separately as a human-readable metric, not part of the score
- **Score matrices saved** so Step 7 evaluation can load them without recomputing

### Next Step: Step 5 — Convert Text to Features (TF-IDF Analysis + Word Embeddings)
- Visualise TF-IDF vocabulary and feature importance per domain
- Explore Word2Vec / GloVe embeddings as an alternative feature representation
- Compare embedding-based similarity scores vs TF-IDF cosine scores